# 03. ROS — VLM 기반 손 추종 / 60분

**선행:** 01 규칙 기반 이동, 02 VLM 프롬프트·응답 검증

**이 노트북의 결과:** `follow_motion()`을 구현하고, 같은 손 좌표·제어기에서 판단만 VLM으로 바꿉니다. TTL 400ms를 유지하며 01의 규칙 동작과 지연·정지 이유를 비교합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

규칙 기반과 같은 손 위치·목표 제어기를 사용하면서 이동 허용 판단만 VLM으로 바꿉니다. 따라서 거북이가 느리거나 멈춘 이유를 인식 좌표·모델 응답·검사·속도 계산으로 나누어 볼 수 있습니다. VLM은 선속도나 각속도를 직접 생성하지 않습니다.

### 전체 과정에서 내 코드가 하는 일

```text
같은 프레임의 이미지·손 중심
  ↓
VLM FOLLOW/STOP
  ↓
ID·시간·pose 검사 TODO
  ↓
공통 목표 제어
  ↓
최종 action·overlay
```

판단과 관측을 연결하고 방향 오차·거리·속도 상한을 구현합니다. 모델 STOP과 검증 거부 STOP을 구분하며 규칙과 VLM을 동시에 같은 거북이에 연결하지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** VLM이 FOLLOW인데 최종 action이 STOP이면 프로그램 오류인가요?

**설명:** 판단 이후에 만료·손 좌표·pose·도달 검사가 남아 있어 정상일 수 있습니다. 같은 frame_id의 원문·관측·action reason을 연결해 확인합니다. 모델 응답만으로 실제 동작을 추측하지 않습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**4일차 환경:** `nepes-course:physical-ai-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 카메라·GUI는 로컬 화면에서 함께 관찰하고 입력 프로세스는 하나만 실행합니다.

**VLM 실습 준비:** VLM 질의에는 공용 Ollama의 `http://127.0.0.1:11434` API와 `gemma4:e2b-it-qat` 모델이 필요합니다. 해당 실습의 연결 확인에서 실패하면 로컬 Setup 담당자에게 서버·모델 준비를 요청합니다. 순수 Python fixture와 YOLO 실습에는 Ollama가 필요하지 않습니다.

**작업 순서:** 아래 첫 설정 셀의 `ROOT`가 `/workspace`인지 확인 → 현재 노트북이 지정한 `exercises/*.py`를 Jupyter 편집기에서 저장 → `student = load_file(...)` 검증 셀을 다시 실행합니다. 새 커널에는 이전 노트북의 변수나 import가 없으므로 이 노트북 자체의 설정·fixture 셀을 사용합니다. 실행 중 보이는 출력은 학습 확인용이고 배포본에는 저장하지 않습니다. TODO 실패와 카메라·모델 환경 실패를 구분하세요.

## 슬라이드와 함께 진행하기

시작 안내는 [4일차 슬라이드 29쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s034) · [배포 PDF 29쪽](../슬라이드/4일차.pdf#page=29)입니다. 노트북의 설명·TODO·검증·실제 실행을 진행한 뒤, [4일차 슬라이드 30쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s055) · [배포 PDF 30쪽](../슬라이드/4일차.pdf#page=30)로 돌아가 30–37쪽에서 결과를 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.


## 정답 활용 안내

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이번 실습이 전체 흐름에서 맡는 역할

[01 규칙 이동](01_ros_gesture_action.ipynb) → [02 VLM 프롬프트 검증](02_vlm_decision.ipynb) → [03 VLM 손 추종](03_ros_vlm_action.ipynb) → [04 판단 유지시간](04_ros_vlm_latency.ipynb) → [05 YOLO detection·pose](05_yolo_detection_pose.ipynb) → [06 pose 좌표·LLM·ROS](06_ros_pose_llm.ipynb). [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.

**구현 실습과 실제 실행의 관계:** 지정한 `exercises/` 파일에 학생이 알고리즘을 구현하고 노트북 검증 셀로 평가합니다. `solutions/`는 별도 참고 정답이며, 참고 정답 검증만 실행해도 학생 TODO가 완성되는 것은 아닙니다. 카메라·ROS 데모는 제공된 실행용 구현을 사용합니다. 학생 파일 저장이 실행 중인 ROS 노드를 자동으로 바꾸지는 않습니다. 실시간 동작과 자신의 함수 검증을 각각 완료하세요.


## 시간 지도

행동 승인·제어 TODO 20분 → 실패 검증 10분 → VLM 손 추종 20분 → 규칙 비교·기록 10분


## 필요한 앞 내용 — 위치와 핵심을 함께 확인

[02_vlm_decision.ipynb의 parser TODO](02_vlm_decision.ipynb), [01_ros_gesture_action.ipynb의 target TODO](01_ros_gesture_action.ipynb)에 판단·좌표의 기초가 있습니다. 시간 검사는 아래에서 함께 설명하고, 유지 시간 변경 실험은 다음 04에서 진행합니다.

- 판단은 `command,hand_state,frame_id,reason` 네 문자열. command는 FOLLOW/STOP, state는 NON_FIST/FIST/NO_HAND/UNKNOWN. FOLLOW는 NON_FIST와만 조합하며 frame_id는 공백 제거 후 비지 않음, reason은 공백 제거 후 1~160자입니다. 틀린 JSON·중복 key·추가 key는 거부합니다.
- 관측 중심 `[x,y]`는 유효한 한 손의 정규화 좌표이고 `(1+9*x,1+9*(1-y))`로 목표를 만듭니다. mirror=True이면 먼저 x를 1-x로 바꿉니다. None이면 손이 유일하지 않은 상태입니다.
- `0≤now-captured<5000`, `0≤issued-captured<5000`, `0≤now-issued<400`, `0≤now-pose_ms<500`을 모두 만족해야 합니다. 새 tick에서 issued_ms를 갱신하지 않습니다. 목표·판단의 frame_id도 같아야 합니다.

기존 노트북의 변수를 가져오지 않고 아래 fixture 셀이 모두 만듭니다. `follow_motion`에 필요한 parser·좌표 함수를 같은 학생 파일의 보조 함수로 직접 작성할 수 있습니다. 독립적인 참고 구현은 [solutions/hand_controller.py](solutions/hand_controller.py)에 두 보조 함수까지 포함하며 아래 접힌 정답에도 같은 내용을 제공합니다. 이전 TODO 파일이 미완성이라는 이유로 이 실습까지 실행 불능이 될 필요는 없습니다.

In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "09.Pseudo-VLA").is_dir():
            return candidate
    raise RuntimeError("09.Pseudo-VLA를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "09.Pseudo-VLA"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('cv2', 'ultralytics', 'rclpy', 'requests')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## pose 제어와 행동 승인

<!-- procedure:lesson-32-bdbef4:start -->
![VLM 판단을 제어 전에 승인하기](../교재그림/절차/lesson-32.svg)

모델 판단과 최종 행동은 다를 수 있습니다. 최종 STOP은 모델 STOP·검사 거부·도착 등 이유를 함께 읽습니다.
<!-- procedure:lesson-32-bdbef4:end -->


목표와 현재 위치 차이의 `atan2`가 목표 방향입니다. 현재 방향과 차이를 -π–π로 감아 회전합니다.
큰 방향 차이(0.6 rad 이상)면 먼저 회전하며, 작은 차이에서만 전진합니다.
선속도 상한 1.2, 각속도 상한 1.5, 목표 도달 반경 0.15를 사용합니다.
VLM은 이 숫자를 정하지 않습니다. VLM이 FOLLOW라도 잘못된 frame/오래된 관측/만료/pose 중단이면 STOP입니다.
VLM 판정에서 규칙 분류를 정답 필터로 쓰지 않습니다. 두 방식에 공통인 한 손 좌표 유효성은 검사합니다.

### 하드웨어·모델 구성의 실측 한계 — 연속 추종과 정지 계약을 구분합니다

2026-09-08 Jetson Orin의 `gemma4:e2b-it-qat`와 640×480 녹화 손 입력을 사용한 실제 ROS VLM 판단 9회의 지연은 약 **2.59–2.77초** 였습니다. FOLLOW는 관측됐지만 승인 TTL은 **0.4초** 이므로 승인이 만료된 뒤 다음 판단이 도착할 때까지 STOP 구간이 생겼습니다. 단일 프레임 판단과 만료 시 정지 계약을 통과해도 매끄러운 실시간 손 추종을 달성했다는 뜻은 아닙니다.

이는 이 하드웨어·모델·입력 구성의 소수 실측입니다. 모든 Jetson이나 모든 장면의 고정 속도가 아니며 모델 적재, GPU 부하, 입력 크기와 처리 경로에 따라 달라집니다. 카메라 프레임률이나 제어 timer 주파수가 높아도 느린 VLM의 새 판단 생성 속도가 빨라지는 것은 아닙니다.

자신의 실행에서는 판단 지연과 FOLLOW/STOP 구간, 목표 도달·위치 오차·주먹 전환 후 정지 반응을 따로 기록하세요. 동작을 이어 보이게 하려고 TTL만 늘리면 오래된 판단으로 이동하는 시간이 길어집니다. 먼저 모델·입력 크기·판단 주기를 재측정하고, 빠른 위치 제어와 느린 의미 판단의 역할 분리를 검토합니다. TTL 변경은 정지 반응도 함께 평가해야 합니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.hand_follow import rule_decision
observation = dict(frame_id="f1", captured_ms=1000, center=[1.0, 0.5], state="NON_FIST")
decision = rule_decision(observation)
pose = (5.5, 5.5, 0.0)
print(observation, decision, pose)


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## TODO — [exercises/hand_controller.py](exercises/hand_controller.py)

`follow_motion(observation, decision, pose, **kwargs)`를 구현합니다. now_ms/issued_ms/pose_ms와 선택 max_age_ms/ttl_ms/pose_age_ms/mirror를 받습니다.
반환은 `command,linear_x,angular_z,reason,target`입니다. 실패는 속도 0, target=None인 STOP입니다.
검사 순서: 입력·숫자 → 촬영 age → 승인 TTL → pose age → strict JSON → frame 일치 → 한 손 좌표 → FOLLOW → 목표·pose → 속도.
이 노트북의 독립적인 참고 정답은 `solutions/hand_controller.py`에 있으며 아래 정답 셀과 접힌 코드에도 제공합니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** observation은 `frame_id,captured_ms,center,state`, decision은 parser의 네 필드 dict, pose는 `(x,y,theta)`입니다. 시각 단위는 ms, theta/각오차는 rad입니다. 반환 dict의 target은 추종 때 `[tx,ty]`, 모든 STOP 때 None입니다.
**작성 흐름 힌트:** 먼저 공통 STOP 반환 도우미 → 기존 명세 순서대로 실패 이유를 조기 반환 → 통과했을 때만 target_xy → 거리와 atan2 목표각 → 각오차를 atan2(sin(error),cos(error))로 -π–π 정규화 → 제한된 속도.
**제어 계약:** 거리≤0.15면 target_reached 정지. abs(error)<0.6일 때만 linear=min(1.2,distance)×max(0,cos(error)), 아니면 0. angular는 2×error를 -1.5–1.5로 제한합니다. 촬영 age<5000, 승인 age<400, pose age<500이 기본값이며 **경계와 같으면 만료** 입니다. 미래 시각도 거부합니다.
**손으로 확인:** center=[1,.5],pose=(5.5,5.5,0), 모든 시각1000, frame_id가 맞고 FOLLOW/NON_FIST이면 target=[10,5.5],linear=1.2,angular=0입니다. now-issued가400이면 STOP입니다.
**설명할 지점:** “유효기간을 매 제어 tick마다 갱신하면 오래된 승인이 살아납니다. 같은 승인 시각을 유지하고 새 관측·새 판단이 올 때만 바꿉니다.”
<!-- todo-review:requirements:end -->

### TODO의 실패 분기와 구현 완료 기준

작성 파일은 `/workspace/09.Pseudo-VLA/exercises/hand_controller.py`입니다. `follow_motion(observation,decision,pose,**kwargs)`의 kwargs에서 필수 `now_ms,issued_ms,pose_ms`와 기본 `max_age_ms=5000,ttl_ms=400,pose_age_ms=500,mirror=False`를 받습니다. `now_ms,issued_ms,pose_ms,captured_ms` 및 각 제한은 bool이 아닌 유한한 int/float여야 합니다. 상태 위치는 현재 pose `(x,y,theta)`이고 x·y는 0~11.1, theta는 유한 rad입니다.

정지 반환은 정확히 `{'command':'STOP','linear_x':0.0,'angular_z':0.0,'reason':이유,'target':None}`입니다. 아래 순서로 검사하여 처음 실패한 이유로 반환합니다. 잘못된 입력 때문에 함수를 예외로 끝내지 않도록 ValueError/TypeError/KeyError/IndexError를 `invalid_input` 정지로 처리합니다.

| 순서 | 검사/상황 | reason |
| --- | --- | --- |
| 1 | 세 제한값 중 0 이하가 있음 | invalid_limits |
| 2 | observation 또는 pose가 None | missing_input |
| 3 | now-captured 또는 issued-captured가 [0,max_age_ms) 밖 | stale_or_future_frame |
| 4 | now-issued가 [0,ttl_ms) 밖 | expired |
| 5 | now-pose_ms가 [0,pose_age_ms) 밖 | stale_pose |
| 6 | decision을 JSON 네 필드 계약으로 재검사했을 때 실패 | invalid_decision |
| 7 | decision과 observation의 frame_id가 다름 | wrong_frame |
| 8 | observation의 center가 None | no_unique_hand |
| 9 | 유효한 decision의 command가 STOP | judgment_stop |
| 10 | 중심→목표 변환과 pose 숫자 검증, pose x/y가 [0,11.1] 밖 | invalid_pose(범위), invalid_input(형식) |
| 11 | 목표까지 거리 ≤0.15 | target_reached |
| 통과 | 제한 속도와 목표 반환 | tracking |

`observation['state']`는 규칙의 분류 결과이고 VLM의 정답 필터로 사용하지 않습니다. 예를 들어 중심이 유효한 UNKNOWN 관측에 모델이 NON_FIST/FOLLOW를 제안하면 다른 조건을 검사한 뒤 추종할 수 있습니다. 한 손이 아닌 경우는 center=None으로 공통 차단합니다.

통과 때 `dx=tx-x`, `dy=ty-y`, `distance=hypot(dx,dy)`, `error=atan2(sin(atan2(dy,dx)-theta),cos(atan2(dy,dx)-theta))`를 구합니다. `abs(error)<0.6`에서만 `linear=min(1.2,distance)*max(0,cos(error))`, 나머지는 linear=0입니다. `angular=2*error`를 [-1.5,1.5]로 제한합니다. 반환은 command=FOLLOW, reason=tracking, target=[tx,ty]이며 제자리 회전도 FOLLOW일 수 있습니다.

**성공 기준:** 오른쪽 목표는 선속도>0·각속도0, 반대 방향은 선속도0·제한 회전, 도달은 STOP입니다. 모든 STOP은 두 속도0·target=None이고 잘못된 frame·400ms 경계 만료·500ms pose 경계·5000ms 관측 경계도 차단됩니다. `decision`·`observation`을 변경하지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_controller", "exercises/hand_controller.py")
def trial(obs=observation, dec=decision, turtle=pose, now=1100):
    return student.follow_motion(obs, dec, turtle, now_ms=now, issued_ms=1050, pose_ms=1100)
assert trial()["linear_x"] > 0
assert trial(turtle=(10, 5.5, 0))["command"] == "STOP"
assert trial(dec={**decision, "frame_id": "other"})["command"] == "STOP"
assert trial(now=1450)["command"] == "STOP"
assert trial(dec={**decision, "command":"STOP", "hand_state":"FIST"})["linear_x"] == 0
import math
# 목표가 (10, 5.5)이고 방향 오차가 0이면 거리 비례 속도에 1.2 상한을 적용합니다.
# 가까운 거리 두 개는 비례 관계, 먼 거리 두 개는 상한을 각각 확인합니다.
for distance, expected_speed in ((0.3, 0.3), (0.6, 0.6), (1.2, 1.2), (4.5, 1.2)):
    action = trial(turtle=(10-distance, 5.5, 0.0))
    assert action["command"] == "FOLLOW" and action["target"] == [10.0, 5.5]
    assert math.isclose(action["linear_x"], expected_speed, rel_tol=1e-9, abs_tol=1e-9), (distance, action)
    assert math.isclose(action["angular_z"], 0.0, abs_tol=1e-9)
# 방향이 약간 다르면 cos 보정을 적용한 뒤에도 선속도 상한 안에 있어야 합니다.
angled = trial(turtle=(5.5, 5.5, 0.3))
assert math.isclose(angled["linear_x"], 1.2*math.cos(0.3), rel_tol=1e-9)
assert 0 <= angled["linear_x"] <= 1.2
turn = trial(turtle=(5.5, 5.5, math.pi))
assert turn["command"] == "FOLLOW" and turn["linear_x"] == 0
assert abs(turn["angular_z"]) <= 1.5
base = dict(observation=observation, decision=decision, pose=pose,
            now_ms=1100, issued_ms=1050, pose_ms=1100)
stop_cases = [
    (dict(ttl_ms=0), "invalid_limits"),
    (dict(pose=None), "missing_input"),
    (dict(now_ms=6000, issued_ms=5900, pose_ms=6000), "stale_or_future_frame"),
    (dict(now_ms=1450, pose_ms=1450), "expired"),
    (dict(pose_ms=600), "stale_pose"),
    (dict(decision={**decision, "speed": 1}), "invalid_decision"),
    (dict(decision={**decision, "frame_id": "other"}), "wrong_frame"),
    (dict(observation={**observation, "center": None}), "no_unique_hand"),
    (dict(decision={**decision, "command": "STOP", "hand_state": "FIST"}), "judgment_stop"),
    (dict(pose=(12, 5.5, 0)), "invalid_pose"),
    (dict(pose=(10, 5.5, 0)), "target_reached"),
    (dict(now_ms=True), "invalid_input"),
]
for changes, expected_reason in stop_cases:
    action = student.follow_motion(**{**base, **changes})
    assert action == dict(command="STOP", linear_x=0.0, angular_z=0.0,
                          reason=expected_reason, target=None), (changes, action)
# 유효한 좌표는 공통으로 검사하고 규칙 state를 VLM의 정답 필터로 쓰지 않습니다.
assert student.follow_motion(**{**base, "observation": {**observation, "state": "UNKNOWN"}})["command"] == "FOLLOW"
print("PASS follow gate, distance-proportional speed and linear/angular limits")


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 참고 정답

강사가 안내하는 시점에 아래 접힌 정답 셀을 여세요. 셀은 `solutions/`의 독립적인 참고 구현이 같은 계약을 통과하는지 확인합니다. 바로 아래 정답 해설·접힌 코드에 입력 검사·경계 처리·반환값을 다시 제공합니다. 실제 카메라·ROS 공통 구현은 별도 runtime이며 참고 정답 실행과 학생 TODO 검증은 각각 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_controller", "solutions/hand_controller.py")
assert reference.follow_motion(observation, decision, pose, now_ms=1100, issued_ms=1050, pose_ms=1100)["linear_x"] > 0
assert reference.follow_motion(observation, decision, pose, now_ms=1450, issued_ms=1050, pose_ms=1450)["command"] == "STOP"
print("PASS reference")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/hand_controller.py](solutions/hand_controller.py)의 `follow_motion`입니다. 먼저 정지 결과를 준비하고 frame·승인·pose 시각, JSON 판정, 동일 frame_id, 단일 손 좌표를 모두 확인한 뒤에만 이동을 계산합니다. 한 검사라도 실패하면 reason을 담은 STOP입니다.

관측 age는 촬영 시각, 승인 TTL은 issued_ms, pose age는 pose_ms로 각각 계산합니다. issued_ms=1000, TTL=400일 때 now=1399는 시간 조건 안이고 1400은 만료입니다. timer가 도는 시각으로 issued_ms를 덮어쓰면 과거 승인을 무한 연장하므로 안 됩니다.

유효한 목표에 대해서 현재 pose와의 거리·방향 오차를 계산하고 각도 오차를 감싼 뒤 제한된 속도를 정합니다. 목표 도달도 정상적인 STOP입니다. VLM은 FOLLOW/STOP을 정하고 속도는 이 공통 제어기가 계산합니다.

**확인 질문:** 약 2.6초 걸리는 VLM과 0.4초 승인 TTL 조합에서 왜 정지 구간이 생기나요? TTL을 늘려 숨기지 말고 이동·정지·지연을 따로 해석하세요.

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/hand_controller.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 관측·판단·자세를 검증한 후 목표 방향과 제한된 속도를 계산합니다.
필요한 좌표 변환과 JSON 검증 함수도 이 파일에 포함했습니다.
관측/승인/자세의 시간, frame_id, 목표 도달을 먼저 검사하며 실패는 STOP입니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
import json
import math

def number(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError('finite number required')
    return float(value)


def target_xy(center, *, mirror=False):
    """Raw camera x right / y down → turtlesim interior [1,10]², y up."""
    if not isinstance(center, (list, tuple)) or len(center) != 2:
        raise ValueError('center must have x,y')
    x, y = map(number, center)
    if not 0 <= x <= 1 or not 0 <= y <= 1:
        raise ValueError('center outside image')
    # x의 선택적 mirror와 y의 위아래 반전을 분리하고 내부 범위 1~10으로 옮깁니다.
    return (1 + 9*(1-x if mirror else x), 1 + 9*(1-y))


HAND_SCHEMA = {
    'type': 'object',
    'properties': {
        'command': {'type': 'string', 'enum': ['FOLLOW', 'STOP']},
        'hand_state': {'type': 'string', 'enum': ['FIST', 'NON_FIST', 'NO_HAND', 'UNKNOWN']},
        'frame_id': {'type': 'string', 'minLength': 1},
        'reason': {'type': 'string', 'minLength': 1, 'maxLength': 160},
    },
    'required': ['command', 'hand_state', 'frame_id', 'reason'],
    'additionalProperties': False,
}


def parse_hand_decision(raw):
    # dict로 변환된 뒤에는 중복 key가 사라지므로 JSON을 읽는 순간 검사합니다.
    def unique_object(pairs):
        value = {}
        for key, item in pairs:
            if key in value:
                raise ValueError(f'duplicate JSON key: {key}')
            value[key] = item
        return value

    try:
        value = json.loads(raw, object_pairs_hook=unique_object,
                           parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))
        if not isinstance(value, dict) or set(value) != set(HAND_SCHEMA['required']):
            return None
        if any(not isinstance(v, str) for v in value.values()):
            return None
        if value['command'] not in ('FOLLOW', 'STOP') or value['hand_state'] not in HAND_SCHEMA['properties']['hand_state']['enum']:
            return None
        if not value['frame_id'].strip() or not 1 <= len(value['reason'].strip()) <= 160:
            return None
        # 형식이 맞아도 FOLLOW와 주먹 상태가 모순되면 행동을 승인할 수 없습니다.
        if value['command'] == 'FOLLOW' and value['hand_state'] != 'NON_FIST':
            return None
        return value
    except (ValueError, TypeError):
        return None


def follow_motion(observation, decision, pose, *, now_ms, issued_ms, pose_ms,
                  max_age_ms=5000, ttl_ms=400, pose_age_ms=500, mirror=False):
    """Recompute every tick from the SAME approved observation; expiry is never refreshed."""
    stop = lambda reason: dict(command='STOP', linear_x=0.0, angular_z=0.0, reason=reason, target=None)
    try:
        now, issued, pose_time = map(number, (now_ms, issued_ms, pose_ms))
        limits = list(map(number, (max_age_ms, ttl_ms, pose_age_ms)))
        if any(v <= 0 for v in limits):
            return stop('invalid_limits')
        if observation is None or pose is None:
            return stop('missing_input')
        captured = number(observation['captured_ms'])
        if not 0 <= now-captured < max_age_ms or not 0 <= issued-captured < max_age_ms:
            return stop('stale_or_future_frame')
        # 승인 시각 issued를 현재 tick으로 갱신하지 않습니다. 경계와 같아도 만료입니다.
        if not 0 <= now-issued < ttl_ms:
            return stop('expired')
        if not 0 <= now-pose_time < pose_age_ms:
            return stop('stale_pose')
        valid = parse_hand_decision(json.dumps(decision))
        if valid is None:
            return stop('invalid_decision')
        if valid['frame_id'] != observation['frame_id']:
            return stop('wrong_frame')
        # VLM owns fist semantics in VLM mode. Geometric UNKNOWN can still have valid center.
        if observation['center'] is None:
            return stop('no_unique_hand')
        if valid['command'] != 'FOLLOW':
            return stop('judgment_stop')
        tx, ty = target_xy(observation['center'], mirror=mirror)
        x, y, theta = map(number, pose)
        if not 0 <= x <= 11.1 or not 0 <= y <= 11.1:
            return stop('invalid_pose')
        distance = math.hypot(tx-x, ty-y)
        if distance <= 0.15:
            return stop('target_reached')
        # -pi/pi 경계를 가로지를 때도 가장 짧은 회전 방향으로 오차를 정규화합니다.
        error = math.atan2(math.sin(math.atan2(ty-y, tx-x)-theta), math.cos(math.atan2(ty-y, tx-x)-theta))
        # 방향 차가 크면 회전만 하고, 작을 때도 선속도/각속도 상한을 지킵니다.
        return dict(command='FOLLOW', linear_x=min(1.2, distance)*max(0, math.cos(error)) if abs(error)<0.6 else 0.0,
                    angular_z=max(-1.5, min(1.5, 2*error)), reason='tracking', target=[tx, ty])
    except (ValueError, TypeError, KeyError, IndexError):
        return stop('invalid_input')
```

</details>
<!-- todo-review:answer:end -->


## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


새 Terminal에서는 아래 블록의 첫 줄 bash부터 전체를 복사합니다. A·B·C 모두 같은 배정 Domain(예:21)을 사용합니다. 노트북 Python 셀에 붙이지 않습니다. 이미 환경을 설정한 같은 B에서 모드·TTL만 바꿀 때는 Ctrl+C 후 launch 명령만 실행합니다.

**처음 준비 또는 패키지를 찾지 못할 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA/scripts/build_workspace.sh
```

**A — 카메라 입력:** 이미 A가 실행 중이면 그대로 유지합니다. 카메라 publisher는 하나만 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

카메라 번호는 배정값입니다. 녹화 입력으로 비교할 때는 A를 Ctrl+C로 종료하고 같은 A에서 아래 명령을 실행합니다. 매 조건에서 같은 시작 지점으로 재생을 다시 시작합니다.
```bash
ros2 run nepes_ros_lab recorded_image_publisher --ros-args -p image_source:=/workspace/Data/videos/07.Mediapipe/person_open_palm.avi
```

**화면 읽기:** /vla/image_overlay의 초록색 점은 전진·회전 중, 빨간색 점은 좌표가 있지만 정지 중입니다. 좌표가 없으면 NO TARGET입니다. Target은 거북이 목표 좌표, age는 제어에 사용한 프레임 경과 시간입니다. 승인 경과 시간·TTL·남은 유효시간도 확인합니다. VLM 추론 중에는 이전 관측 화면이 유지됩니다. 최신 원본은 /camera/image_raw입니다. mirror:=true는 목표 x만 반전하고 영상과 점은 원본 위치를 유지합니다.

**B — VLM 기반 이동:** 01의 B를 Ctrl+C로 종료합니다. 새 B에서도 실행 가능하도록 환경 설정을 포함했습니다. 공용 Ollama에 gemma4:e2b-it-qat가 있어야 합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=vlm \
  ttl_ms:=400 \
  vlm_model:=gemma4:e2b-it-qat \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA/03-hand-vlm.jsonl
```

01과 같은 입력·초기 pose·mirror=false·TTL 400ms를 사용합니다. 여기서는 TTL을 바꾸지 않습니다.

**C — 관측·판단·행동 확인:** 새 Terminal에서 전체를 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 topic echo /vla/action
```

연속 출력은 Ctrl+C로 끝낸 뒤, 같은 C에서 필요에 따라 ros2 topic echo /vla/perception 또는 ros2 topic echo /vla/decision을 실행합니다. frame_id·reason·TTL·승인 경과 시간을 영상과 비교합니다.

**종료:** A 중단 → C에서 STOP 확인 → B 종료 → C 종료. 다음 노트북으로 바로 이어갈 때는 A를 유지하고 B만 종료합니다. GUI 없는 자습은 B launch 앞에 QT_QPA_PLATFORM=offscreen, 인자에 with_image_view:=false를 사용합니다.

학생 TODO는 Python 검증 셀로 평가합니다. 실제 ROS는 제공 공통 구현을 실행하며 학생 파일 변경이 자동 반영되지는 않습니다.


## 완료 검사

1. 01과 같은 손 이동·주먹 전환을 반복하고 FOLLOW와 최종 행동을 구분합니다.
2. 모델 STOP, 형식·frame 거부, 목표 도착, 승인 만료를 로그에서 구분합니다.
3. 03-hand-vlm.jsonl과 01 규칙 로그의 지연·추종·정지 차이를 기록합니다.
4. TTL은 400ms로 유지합니다. 다음 04에서 유지 시간을 바꿉니다.

항상 STOP이면 추종 성공이 아닙니다. 정답률·전환 후 반응은 원본 장면과 로그를 사람이 대조하며 미측정은 미측정으로 남깁니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

[4일차 슬라이드 30쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s055) · [배포 PDF 30쪽](../슬라이드/4일차.pdf#page=30)에서 30–37쪽을 복습합니다. 자신의 TODO 검증과 실제 관측·미측정을 나누어 설명합니다.
<!-- slide-sync:end -->

## 다음 실습 — 유지 시간 비교

[04 VLM 지연 실험](04_ros_vlm_latency.ipynb)에서 모델·입력·제어기를 유지하고 TTL만 바꿉니다.
